# Medical-Chatbot Colab Evaluation (GenerationService)

Pipeline:
1. Load data/raw/VL_..._...json (VL merged dataset file)
2. Split by `q_type` (1, 2, 3)
3. Mode A (`LLM only`) generation with Qwen2.5-Instruct
4. Mode B (`LLM + RAG`) generation with Qwen2.5-Instruct + `chromadb/`
5. Evaluate with `src/evaluation/metrics.py`


In [ ]:
# ===== 0) Install dependencies (first run only) =====
!pip -q install -U pip
!pip -q install -r requirements.txt


In [ ]:
# ===== 1) Paths and runtime settings =====
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
SRC_PATH = REPO_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

DATA_PATH = REPO_ROOT / 'data/raw/VL_내과_통합.json'
CHROMA_PATH = REPO_ROOT / 'chroma_db'
OUTPUT_DIR = REPO_ROOT / 'outputs/colab_eval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f'Missing data file: {DATA_PATH}'
assert CHROMA_PATH.exists(), f'Missing ChromaDB path: {CHROMA_PATH}'
print('REPO_ROOT =', REPO_ROOT)
print('DATA_PATH =', DATA_PATH)
print('CHROMA_PATH =', CHROMA_PATH)

In [ ]:
# ===== 2) Load data and split by q_type =====
import json
import pandas as pd

rows = json.loads(DATA_PATH.read_text(encoding='utf-8'))
df = pd.DataFrame(rows)
df['q_type'] = df['q_type'].astype(int)

required_cols = ['qa_id', 'q_type', 'question', 'answer']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f'Missing required column: {col}')

split_by_qtype = {q: g.reset_index(drop=True) for q, g in df.groupby('q_type')}
for q in [1, 2, 3]:
    print(f'q_type={q}:', len(split_by_qtype.get(q, pd.DataFrame())))

df.head(2)

In [ ]:
# ===== 3) Initialize GenerationService and RAG helpers =====
from config.settings import settings
from llm.generation_service import GenerationService
from retrieval.dense.embedder import Embedder
from retrieval.dense.chroma_store import ChromaStore

settings.model_backend = 'qwen'
settings.model_path = 'Qwen/Qwen2.5-7B-Instruct'
settings.model_mode = 'A'
settings.max_new_tokens = 256
settings.temperature = 0.2

# Retrieval embedding model.
# If OPENAI_API_KEY is missing, Embedder falls back to local sentence-transformers model.
settings.embedding_model = os.getenv('EMBEDDING_MODEL', 'jhgan/ko-sroberta-multitask')
settings.openai_api_key = os.getenv('OPENAI_API_KEY', settings.openai_api_key)

gen_service = GenerationService()
embedder = Embedder(embedding_model=settings.embedding_model, api_key=settings.openai_api_key)
store = ChromaStore(collection_name=settings.chroma_collection, persist_directory=str(CHROMA_PATH))

print('model_path =', settings.model_path)
print('embedding_model =', settings.embedding_model)
print('chroma_collection =', settings.chroma_collection)
print('chroma_count =', store.count())

In [ ]:
# ===== 4) Generation functions for Mode A / Mode B =====
from typing import Optional
from tqdm.auto import tqdm

def build_rag_context(question: str, top_k: int = 5) -> str:
    q_vec = embedder.embed(question)
    results = store.query(query_embedding=q_vec, top_k=top_k)

    ids = (results.get('ids') or [[]])[0]
    docs = (results.get('documents') or [[]])[0]
    metas = (results.get('metadatas') or [[]])[0]
    dists = (results.get('distances') or [[]])[0]

    if not ids:
        return ''

    parts = []
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), 1):
        meta = meta or {}
        source = meta.get('source_spec', 'unknown')
        sim = max(0.0, 1.0 - (dist / 2.0))
        parts.append(f'[Ref {i}] (source: {source}, sim: {sim:.3f})\n{doc}')
    return '\n\n'.join(parts)

def generate_answer(question: str, mode: str = 'A', top_k: int = 5) -> str:
    if mode == 'A':
        return gen_service.generate(query=question, context=None, mode='A')
    if mode == 'B':
        ctx = build_rag_context(question, top_k=top_k)
        return gen_service.generate(query=question, context=ctx, mode='B')
    raise ValueError('mode must be A or B')

def run_generation(df_in, mode: str, top_k: int = 5, sample_n: Optional[int] = None):
    data = df_in.copy()
    if sample_n is not None:
        data = data.head(sample_n).copy()

    preds = []
    for q in tqdm(data['question'].tolist(), desc=f'Generate mode={mode}'):
        try:
            preds.append(generate_answer(q, mode=mode, top_k=top_k))
        except Exception as e:
            preds.append(f'[ERROR] {e}')

    out = data.copy()
    out[f'pred_{mode}'] = preds
    return out

In [ ]:
# ===== 5) Run Mode A and Mode B =====
# For quick checks in Colab, start with a small sample.
SAMPLE_N_PER_QTYPE = 20  # set None for full run
TOP_K = 5

mode_a_parts = []
mode_b_parts = []

for q in [1, 2, 3]:
    part = split_by_qtype.get(q)
    if part is None or len(part) == 0:
        continue

    a_out = run_generation(part, mode='A', top_k=TOP_K, sample_n=SAMPLE_N_PER_QTYPE)
    b_out = run_generation(part, mode='B', top_k=TOP_K, sample_n=SAMPLE_N_PER_QTYPE)

    mode_a_parts.append(a_out[['qa_id', 'q_type', 'question', 'answer', 'pred_A']])
    mode_b_parts.append(b_out[['qa_id', 'q_type', 'pred_B']])

pred_a_df = pd.concat(mode_a_parts, ignore_index=True) if mode_a_parts else pd.DataFrame()
pred_b_df = pd.concat(mode_b_parts, ignore_index=True) if mode_b_parts else pd.DataFrame()

pred_df = pred_a_df.merge(pred_b_df, on=['qa_id', 'q_type'], how='left')
pred_df.to_csv(OUTPUT_DIR / 'predictions_A_B.csv', index=False, encoding='utf-8-sig')
print('saved:', OUTPUT_DIR / 'predictions_A_B.csv')
pred_df.head(3)

In [ ]:
# ===== 6) Evaluate with metrics.py =====
from evaluation.metrics import exact_match, rouge_l, bert_score

def evaluate_with_metrics(df_eval, pred_col: str):
    refs = df_eval['answer'].fillna('').tolist()
    preds = df_eval[pred_col].fillna('').tolist()

    ems = [exact_match(p, r) for p, r in zip(preds, refs)]
    rls = [rouge_l(p, r) for p, r in zip(preds, refs)]
    bss = bert_score(preds, refs)

    scored = df_eval.copy()
    scored['exact_match'] = ems
    scored['rouge_l'] = rls
    scored['bert_score'] = bss

    summary_rows = []
    for q in [1, 2, 3]:
        sub = scored[scored['q_type'] == q]
        if len(sub) == 0:
            continue
        summary_rows.append({
            'mode': pred_col.replace('pred_', ''),
            'q_type': q,
            'n': len(sub),
            'exact_match': float(sub['exact_match'].mean()),
            'rouge_l': float(sub['rouge_l'].mean()),
            'bert_score': float(sub['bert_score'].mean()),
        })

    summary_rows.append({
        'mode': pred_col.replace('pred_', ''),
        'q_type': 'all',
        'n': len(scored),
        'exact_match': float(scored['exact_match'].mean()),
        'rouge_l': float(scored['rouge_l'].mean()),
        'bert_score': float(scored['bert_score'].mean()),
    })

    return scored, pd.DataFrame(summary_rows)

scored_A, summary_A = evaluate_with_metrics(
    pred_df[['qa_id', 'q_type', 'question', 'answer', 'pred_A']].copy(),
    'pred_A'
)
scored_B, summary_B = evaluate_with_metrics(
    pred_df[['qa_id', 'q_type', 'question', 'answer', 'pred_B']].copy(),
    'pred_B'
)

summary_all = pd.concat([summary_A, summary_B], ignore_index=True)
summary_all.to_csv(OUTPUT_DIR / 'summary_metrics_A_B.csv', index=False, encoding='utf-8-sig')
scored_A.to_csv(OUTPUT_DIR / 'scored_mode_A.csv', index=False, encoding='utf-8-sig')
scored_B.to_csv(OUTPUT_DIR / 'scored_mode_B.csv', index=False, encoding='utf-8-sig')

print('saved:', OUTPUT_DIR / 'summary_metrics_A_B.csv')
print('saved:', OUTPUT_DIR / 'scored_mode_A.csv')
print('saved:', OUTPUT_DIR / 'scored_mode_B.csv')
summary_all

## Notes
- Set `SAMPLE_N_PER_QTYPE = None` for full evaluation.
- RAG quality depends on whether query embedding model matches the model used when building `chromadb/`.
- If Colab GPU memory is not enough, switch to a smaller Qwen model (for example, 3B).
